# Data quality
Validate essential rules and record their results.


In [ ]:
import importlib
import json
import sys
from html import escape
from pathlib import Path

project_root = Path(dbutils.widgets.getAll().get(
    "project_root", "/Workspace/Users/allanjs3@yahoo.com.br/beverage_sales"
))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
importlib.invalidate_caches()
import data_quality.checks as quality_checks
importlib.reload(quality_checks)


## Execution settings
Manual runs always validate. Automatic runs use the upstream load results.


In [ ]:
dbutils.widgets.text("project_root", str(project_root), "Project Root")
dbutils.widgets.dropdown("execution_mode", "Manual", ["Manual", "Automatic"], "Execution Mode")
dbutils.widgets.text("upstream_results", "{}", "Upstream Results")
dbutils.widgets.text("run_id", "", "Run ID")
mode = dbutils.widgets.get("execution_mode")
changes = dbutils.widgets.get("upstream_results")
run_id = dbutils.widgets.get("run_id")
if not quality_checks.should_validate(mode, changes):
    displayHTML("<h3>Data quality — no data written; validation skipped.</h3>")
    dbutils.notebook.exit(json.dumps({"status": "SKIPPED", "checks": 0}))


## Quality results
Check the loaded tables and save one result per rule and table.


In [ ]:
results = quality_checks.run_quality(spark, project_root, run_id)
failed = [row for row in results if row[4] != "PASS"]
rows = "".join(
    "<tr>" + "".join("<td>" + escape(str(value or "")) + "</td>"
                    for value in (row[2], row[7], row[6], row[4], row[5])) + "</tr>"
    for row in results
)
displayHTML(
    "<style>table.dq{border-collapse:collapse;width:100%;font-family:Arial}"
    ".dq td,.dq th{padding:9px;text-align:left;border-bottom:1px solid #ddd}"
    ".dq th{background:#f3f6fa}</style>"
    f"<h3>Data quality — {len(results) - len(failed)} of {len(results)} checks passed</h3>"
    "<table class='dq'><tr><th>Table</th><th>Table description</th><th>Rule</th><th>Status</th><th>Message</th></tr>"
    + rows + "</table>"
)
if failed:
    raise ValueError(
        f"{len(failed)} quality checks failed. Results saved in {quality_checks.RESULTS_TABLE}."
    )
dbutils.notebook.exit(json.dumps({"status": "PASS", "checks": len(results), "run_id": results[0][0]}))
